# Chattahoochee Trout Forecast — Pipeline Walkthrough

A short, end-to-end demonstration of the forecasting pipeline: pulling real river and weather data, engineering features, constructing the training label, and generating a live forecast from the trained model.

This notebook stops after generating predictions. The Streamlit dashboard (`app/app.py`) and the daily automation (`.github/workflows/daily_forecast.yml`) are separate layers built on top of the same pipeline shown here — see `README.md` and `CODE_DOCUMENTATION.md` for those.

Everything below uses the project's real modules (`src/...`), not simplified stand-ins, and pulls live data from USGS and Open-Meteo when run.

In [ ]:
import sys
sys.path.insert(0, "..")  # project root, so `from src...` imports resolve

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

from src.config import STATIONS

STATION = "buford_dam"  # the process below is identical for medlock_bridge

## 1. Data ingestion

Pull the last few days of real-time river sensor readings from USGS (15-minute resolution), and matching weather from Open-Meteo, for Buford Dam.

In [ ]:
from src.ingestion.usgs import fetch_and_pivot

start = (pd.Timestamp.now() - pd.Timedelta(days=3)).strftime("%Y-%m-%d")
end = (pd.Timestamp.now() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

usgs_df = fetch_and_pivot(STATION, start, end)
usgs_df.tail()

Each row is one 15-minute reading: discharge, water temperature, dissolved oxygen, and specific conductance. Now pull weather for the same window (the forecast endpoint, which also includes a few recent days of actuals).

In [ ]:
from src.ingestion.weather import fetch_weather_forecast, join_weather_to_usgs

station_info = STATIONS[STATION]
weather_df = fetch_weather_forecast(station_info["lat"], station_info["lon"], past_days=3, forecast_days=2)
weather_df.tail()

## 2. Joining river and weather data

Weather is hourly; USGS is every 15 minutes. The hourly weather is forward-filled onto the finer USGS grid rather than interpolated, since rainfall and cloud cover aren't smooth continuous signals.

In [ ]:
joined = join_weather_to_usgs(usgs_df, weather_df)
joined.tail()

## 3. Feature engineering

Turn the raw readings into signals the model can use: discharge rate of change (Buford is a peaking hydro facility, so *how fast* discharge is changing often matters more than the level), rolling temperature averages, season, local time of day, and whether this is a stocked week.

In [ ]:
from src.features.engineering import engineer_features

featured = engineer_features(joined)
featured[["discharge_cfs", "discharge_delta_1hr_cfs", "water_temp_roll_24hr",
          "season", "time_of_day", "stocked_week_flag"]].tail()

## 4. Composite scoring — building the training label

There's no ground-truth "was this a good fishing day" data anywhere, so the training label is built from a station-specific point rubric over water temperature, discharge, discharge stability, and dissolved oxygen (see `CODE_DOCUMENTATION.md` for the exact thresholds). This step is only used to construct **historical** training labels — it is not part of live prediction, since forecasting the future doesn't require scoring the present.

In [ ]:
from src.scoring.composite import score_and_classify

labeled = score_and_classify(featured, STATION)
labeled[["water_temp_c", "discharge_cfs", "composite_score", "condition"]].tail()

## 5. Model training (summary, not re-run here)

Training happens offline against the full 2023-2026 historical record (~105,000 readings per station), expanded to roughly 5 million rows so one model learns every forecast horizon from 1 to 48 hours ahead. That takes a few minutes per station and isn't repeated in this notebook — instead, the already-trained model is loaded directly.

In [ ]:
import json
import xgboost as xgb

model = xgb.XGBClassifier()
model.load_model(f"../models/{STATION}_xgb.json")

spec = json.loads(open(f"../models/{STATION}_feature_spec.json").read())
print("Model trained for horizons 1 to", spec["max_horizon_hours"], "hours ahead")
print("Held-out test period starts:", spec["test_start"])

## 6. Generating a live forecast

This calls the real live-prediction pipeline (`src/pipeline/predict.py`): pull the most recent conditions, run them through the trained model once per hour for the next 48 hours, and summarize into a single daily rating.

In [ ]:
from src.pipeline.predict import predict_station

result = predict_station(STATION)

print("Anchor time (most recent reading used):", result["anchor_time_utc"])
print("Current conditions:", result["current_summary"])
print("Today's overall rating:", result["daily_badge"])

Here's the hourly forecast for the next 12 hours:

In [ ]:
pd.DataFrame(result["hourly_timeline"][:12])

## 7. Viewing the dashboard

The Streamlit app displays exactly the same output shown above (current conditions, daily rating, hourly timeline), read from a saved JSON file rather than recomputed. The cell below starts it as a background process and opens it in your browser -- the notebook stays usable while it runs.

In [ ]:
import subprocess
import time
import webbrowser
from pathlib import Path

project_root = Path("..").resolve()
dashboard_proc = subprocess.Popen(
    ["streamlit", "run", "app/app.py", "--server.headless", "true"],
    cwd=project_root,
)
time.sleep(5)  # give the server a moment to start
webbrowser.open("http://localhost:8501")
print("Dashboard running at http://localhost:8501 -- open that link if it did not open automatically.")

When you're done looking at the dashboard, run the next cell to stop the server (otherwise it keeps running in the background until you shut down the kernel).

In [ ]:
dashboard_proc.terminate()
dashboard_proc.wait()
print("Dashboard stopped.")

That's the full pipeline: real sensor and weather data in, a trained model out, an hour-by-hour forecast, and the same dashboard anyone would see by running `streamlit run app/app.py` themselves. The GitHub Actions workflow runs exactly this prediction logic automatically once a day so the dashboard stays current.